# Flash attention
## 搭建环境
在ubuntu虚拟机上搭建了环境，可以正确编译以及cuda程序，实现cuda程序的自动跳转，方便编写cuda代码
因此编写代码就放在虚拟机上

因为flash attention2 3 需要利用特殊的硬件特性，因此需要在租相应的平台，在平台上搭建环境很麻烦

先看看autodl，能不能采用docker的方式运行
## 实现flash attention
### 实现层次
需要先看看怎么实现，应该是直接实现flash attention 算子，包括对应的forward、backward

上层的transfomer 模块相应的调用算子（看一下以前的transformer实现）

最后确认了实现的层次：Ops -> CUDA Kernel -> Pybind -> Op Wrapper -> Module Call
### 实现方法
应该用什么实现呢？cuda、triton、cuTile

使用cuda编程的话，应该借用cutblass模版库进行编程

如果使用triton实现的话，绕过了pybind，但是将cuda后端管理的ptr给triton模块，然后再在triton层面进行编程

使用cuTile编程的话，依旧是python编程，和triton类似

### 测试程序
需要先编写测试程序，首先是flash attention 算子的测试程序，该测试程序需要包括一下几个方面

1.直接调用cuda程序中的实现进行测试（确定在ops层面进行测试）

2.需要验证算子的正确性，那和什么进行比对呢（或者在python层面进行比对正确性，参考一下以前的比较）

3.每个测试应该进行多次迭代，从而可以测试时间，包括不同seq长度

（可以参考官方的flash attention实现，看看它是怎么进行测试的）





接下来要做的事：

在autodl平台上跑一次

了解cuTile，决定是用cutlass还是用cuTile

看看别人的实现以及怎么测试、怎么benchmark

backward 不应该都是tensor操作吗，直接调用Narray api不就构建不了计算图了吗？（需要构建一个ops用来计算backward）

测试的时候是调用ops测试，还是module呢，以及怎么获得时间


In [ ]:
!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git

# Download the PTB dataset

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

In [6]:
!make clean
!make

rm -rf build python/needle/backend_ndarray/ndarray_backend*.so


CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python: /root/miniconda3/envs/needle/bin/python (found version "3.10.19") found components: Development Interpreter 

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [7]:
import sys
sys.path.append('./python')

## 测试
### 正确性验证
如果在NDArray层面去测试的话，需要借用numpy来进行比较，而numpy没有现有的api计算flash attentionn，需要根据已知的qkv计算self attention，因此需要创建numpy的qkv，然后根据self attention的定义去计算，再与cuda 的NDArray计算结果进行比较，比较麻烦。

因此本测试在flash attention module层面进行正确性验证，仿照已有测试中的attention_activation 测试编写，将flash attention的结果与已知的label进行对比，同时编写了新的测试将结果和torch的flash attention 计算结果对比，在与torch对比的测试用例 sequence length比较长，符合实际。
### benchmark
对于benchmark说，需要计算TFLOPS，为了最贴近计算，采用算子层面进行benchmark，先对GPU进行预热，flashattention进行计算，迭代50次，计算时间以及总的操作数，从而计算TFOPS。当前的计算基于causal = false，dropout = 0

In [ ]:
!python3 -m pytest tests/hw4/test_transformer.py -l -v -k "attention_activation_vs_torch"

In [9]:
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and 64 and False and 0.0 and cuda"

============================= test session starts ==============================
platform linux -- Python 3.10.19, pytest-9.0.2, pluggy-1.6.0 -- /root/miniconda3/envs/needle/bin/python3
cachedir: .pytest_cache
rootdir: /root/needle
collected 96 items / 88 deselected / 8 selected                                

tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-128-5-2] PASSED [ 12%]
tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-128-10-2] PASSED [ 25%]
tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-256-5-2] PASSED [ 37%]
tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-256-10-2] PASSED [ 50%]
tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-1024-5-2] PASSED [ 62%]
tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-1024-10-2] PASSED [ 75%

In [8]:
!python3 -m pytest tests/project/test_flashattention.py  -s -l -v -k "test_attention_activation_vs_torch and 5 and 1024 and 64 and False and 0.0 and cuda"

============================= test session starts ==============================
platform linux -- Python 3.10.19, pytest-9.0.2, pluggy-1.6.0 -- /root/miniconda3/envs/needle/bin/python3
cachedir: .pytest_cache
rootdir: /root/needle
collecting ... 


libgomp: Invalid value for environment variable OMP_NUM_THREADS
Using needle backend
collected 96 items / 95 deselected / 1 selected                                

tests/project/test_flashattention.py::test_attention_activation_vs_torch[cuda-0.0-False-64-1024-5-2] flash attention kernel (batch_size=2, num_heads=5, q_len=1024, kv_len=1024, head_dim=64, dropout=0, causal=false)
smem_ptr[32b](0x7f551100a000) o ((_2,_2),_1,_8):((_64,_8),_0,_512)ptr[32b](0x7f550ffffba0) o ((_2,_2),_1,_8):((_1,_2),_0,_4)smem_ptr[32b](0x7f551100a000) o ((_2,_2),_1,_8):((_64,_8),_0,_512)ptr[32b](0x7f550ffffc20) o ((_2,_2),_1,_8):((_1,_2),_0,_4)ptr[16b](0x7f550ffffca0) o ((_2,_2),_1,_8):((_1,_2),_0,_4)torch.float16
PASSED

======================= 1 passed, 95 deselected in 1.71s =======================


In [ ]:
# stub hook 验证
import sys
!{sys.executable} -m pytest tests/project/test_flashattention_stub.py -l -v

In [10]:
# 计算TFLOPS
!python3 tests/project/benchmark.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

Using needle backend

libgomp: Invalid value for environment variable OMP_NUM_THREADS
Benchmarking Flash Attention: B=8, H=12, L=1024, D=64, Causal=False, Dropout=0.0
Warming up for 10 iterations...
flash attention kernel (batch_size=8, num_heads=12, q_len=1024, kv_len=1024, head_dim=64, dropout=0, causal=false)
smem_ptr[32b](0x7fbed900a000) o ((_2,_2),_1,_8):((_64,_8),_0,_512)ptr[32b](0x7fbed7fffba0) o ((_2,_2),_1,_8):((_1,_2),_0,_4)smem_ptr[32b](0x7fbed900a000) o ((_2,_2),_1,_8):((_64,_8),_0,_512)ptr[32b](0x7fbed7fffc20) o ((_2,_2),_1,_8):((_1,_2),_0,_4)ptr[16b](0x7fbed7fffca0) o ((_2,_2),_1,_8):((_1,_2),_0,_4)flash attention kernel (batch_size=8, num_heads=12, q_len=1024, kv_len=1024, head_dim=64, dropout=0, causal=false)
smem_ptr[32b](0x7fbed900a000) o ((_2,_2),_1,_8):((_64,_8),_0,_512)ptr[32b](0x7fbed7fffba0) o ((_2,_2),_1,_8):((_1,_2),_0,_4)smem_ptr[32b](0x7fbed900a000) o ((_2,_2),_1,_8):((_64,_8),_0,_512)ptr[32b](0x7fbed7fffc20) o ((_2,_2),_1,_8):((_1,_2),_0,_4)ptr[16b](0x7fbed7

In [ ]:
# baseline
!python3 tests/project/benchmark_torch.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# ncu profile
!ncu --launch-skip 10 --launch-count 1

## stub
对于ops层面的stub来说，是返回tensor tuple还是tensor呢？
* 需要查看probs对于后续有没有什么作用，如果没有作用可以直接返回result tensor
* 如果返回tensor tuple，是backend计算直接返回tensor tuple还是在ops层进行组装后返回，需要参考stack的实现，返回tensor tuple后进行第i个结果的取用会不会产生额外的开销

最后决定只是返回result，因此定义为tensorops


## 调用层次关系
module调用ops进行计算，ops的forward计算的是NDArray，使用NDArray定义的array api进行计算，NDArray的backend device不同，因此调用进不同的device function。

## 精度
在pytorch中的attention的实现中，因为A100的tensor core只接受输入的形式为 fp16/bf16/tf32，不支持fp32的输入，因此没有开启混合精度的情况下，是没有使用flash attention的实现的，而使用的是原生的的其他优化过的kernel

如果开启混合精度模式，计算的流程如下：QKV（fp32）加载到sram中cast 为（fp16），使用tensor做矩阵乘法时，结果accumulator保存为fp32。进行softmax的时候，为了防止溢出保持fp32，再cast为fp16与 V（fp16）相乘，
最后的结果O为fp16（为了节省sram 写到 GMEM中的带宽）。

现在我自己的实现最好确定输入的类型时fp16，那这样其他实现的cuda计算就用不了了，一个weired解决办法是输入继续保持为fp32但是计算的结果o为fp32再重新cast为fp16。

* 看看别人的flash attention怎么实现的
    * 别人的flash attention中没有使用cute模版编程，只是借用了cutlass的gemm实现来优化自己的设计。

* 如果在我的实现中使用cutlass，应该怎么使用，和cute有区别吗？
    * CuTe 强大的 Layout Algebra (布局代数) 能够让你在不陷入指针算术泥潭的情况下，优雅地处理复杂的 Tensor Core 数据映射、Shared Memory Swizzle 和 Bank Conflict。


1. 现在需要研究cute 的核心，使用cute 模版编程来实现自己的flash attention。


